# Plant-disease edge project — training on Colab / Kaggle

Runs the full experiment from the proposal, in order:

1. Data: PlantVillage tomato (lab) + PlantDoc tomato (field)
2. Train MobileNetV3-Small and EfficientNet-Lite0 (standard augmentation) → **lab vs field gap**
3. Train with strong augmentation, then fine-tune on a few field images → **recovered field accuracy**
4. Convert to TFLite FP32 + INT8 → **FP32 vs INT8 accuracy**
5. Save everything to Google Drive; copy `models/*.tflite` to the Pi and run `scripts/benchmark_pi.py` there

**Runtime → Change runtime type → GPU (T4).** Total time is roughly 1.5–2.5 h.

## 0. Get the project folder
Upload the `plant_disease_edge` folder (without `.venv`) to **My Drive**, then run:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/plant_disease_edge'
!rsync -a --exclude .venv --exclude data {DRIVE}/ /content/plant_disease_edge/
%cd /content/plant_disease_edge
# keep Colab's TensorFlow; install the matching tf_keras + the rest
import tensorflow as tf
TFV = '.'.join(tf.__version__.split('.')[:2])
!pip -q install "tf_keras~={TFV}.0" tensorflow_hub scikit-learn pyyaml paho-mqtt kaggle

## 1. Data
Needs a Kaggle API token: kaggle.com → Settings → *Create New Token* → upload `kaggle.json` when asked.
Team field photos (optional): put them in `data/own_photos/<class>/` (e.g. `data/own_photos/early_blight/`) **before** this step.

In [ ]:
import os
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    from google.colab import files
    up = files.upload()  # choose kaggle.json
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    os.replace('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
!rsync -a {DRIVE}/data/own_photos/ data/own_photos/ 2>/dev/null; true
!python scripts/prepare_data.py --download

## 2. Baselines on lab images (standard augmentation)

In [ ]:
!python scripts/train.py --model mobilenetv3small --augment standard
!python scripts/train.py --model efficientnet_lite0 --augment standard

### Domain gap: lab test vs field test

In [ ]:
!python scripts/evaluate.py models/mobilenetv3small_standard models/efficientnet_lite0_standard

## 3. Close the gap: strong augmentation, then fine-tune on a few field images

In [ ]:
!python scripts/train.py --model mobilenetv3small --augment strong
!python scripts/train.py --model efficientnet_lite0 --augment strong
!python scripts/finetune_field.py --base models/mobilenetv3small_strong
!python scripts/finetune_field.py --base models/efficientnet_lite0_strong
!python scripts/evaluate.py models/mobilenetv3small_strong models/efficientnet_lite0_strong \
                            models/mobilenetv3small_field models/efficientnet_lite0_field

## 4. Quantize to TFLite (FP32 + INT8) and check accuracy

In [ ]:
for m in ['mobilenetv3small_standard', 'mobilenetv3small_field',
          'efficientnet_lite0_standard', 'efficientnet_lite0_field']:
    !python scripts/quantize.py models/{m}
!python scripts/evaluate.py models/*.tflite

## 5. Results table

In [ ]:
import pandas as pd
df = pd.read_csv('results/summary.csv')
df = df.drop_duplicates(['model', 'set'], keep='last')
df.pivot(index='model', columns='set', values=['accuracy', 'macro_f1']).round(3)

In [ ]:
from IPython.display import Image, display
for f in ['results/mobilenetv3small_standard_keras__field_cm.png',
          'results/mobilenetv3small_field_int8__field_cm.png']:
    if os.path.exists(f): display(Image(f, width=520))

## 6. Save to Drive
Then copy `models/*.tflite` to the Pi's `plant_disease_edge/models/` folder (USB stick or `scp`).

In [ ]:
!mkdir -p {DRIVE}/models {DRIVE}/results
!cp models/*.tflite models/*_history.json {DRIVE}/models/
!cp -r results/* {DRIVE}/results/
# small copy of the test sets so the Pi can measure on-device accuracy (benchmark_pi.py --accuracy)
!tar czf {DRIVE}/test_sets.tgz data/lab/test data/field/test
!ls -la {DRIVE}/models